# Lab 3 · Exercise A — On/off with a Gaussian constraint
Two ways to model the uncertainty on the background $b$:
an **auxiliary measurement** (Poisson, as in Lab 2) or a **constraint term** (Gaussian, from “$3.8 \\pm 0.6$”).
We compute the significance of the top-quark excess ($n_{\\rm on}=17$, $n_{\\rm off}=40$, $\\tau=10.53$) with both.

In [ ]:
import ROOT
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## A1 · Two model builders
`build_poisson_aux` is given. Complete `build_gauss`. Both return a dictionary that keeps all RooFit objects alive.

In [ ]:
def build_poisson_aux(non, noff, tau_val):
    """On/off model: Pois(n_on | s+b) * Pois(n_off | tau*b)  (auxiliary measurement)."""
    s   = ROOT.RooRealVar("s", "signal", max(non - noff / tau_val, 0.5), 0, 200)
    b   = ROOT.RooRealVar("b", "background", max(noff / tau_val, 0.5), 0.01, 200)
    tau = ROOT.RooRealVar("tau", "tau", tau_val); tau.setConstant(True)
    n_on  = ROOT.RooRealVar("n_on", "n_on", 0, 300)
    n_off = ROOT.RooRealVar("n_off", "n_off", 0, 600)
    mu_on  = ROOT.RooFormulaVar("mu_on", "@0+@1", ROOT.RooArgList(s, b))
    mu_off = ROOT.RooFormulaVar("mu_off", "@0*@1", ROOT.RooArgList(tau, b))
    p_on  = ROOT.RooPoisson("p_on", "p_on", n_on, mu_on)
    p_off = ROOT.RooPoisson("p_off", "p_off", n_off, mu_off)
    pdf   = ROOT.RooProdPdf("pdf", "pdf", ROOT.RooArgList(p_on, p_off))
    data  = ROOT.RooDataSet("d", "d", ROOT.RooArgSet(n_on, n_off))
    n_on.setVal(non); n_off.setVal(noff)
    data.add(ROOT.RooArgSet(n_on, n_off))
    nll = pdf.createNLL(data)
    return dict(locals())      # keep every RooFit object alive

In [ ]:
def build_gauss(non, b0_val, sigma_b):
    """Counting model with a Gaussian constraint on b:  Pois(n_on | s+b) * Gauss(b0 | b, sigma_b)."""
    s  = ROOT.RooRealVar("s", "signal", max(non - b0_val, 0.5), 0, 200)
    b  = ROOT.RooRealVar("b", "background", b0_val, 0.01, 200)
    b0 = ROOT.RooRealVar("b0", "global observable", b0_val, 0, 200); b0.setConstant(True)
    sb = ROOT.RooRealVar("sb", "sigma_b", sigma_b);                  sb.setConstant(True)
    n_on = ROOT.RooRealVar("n_on", "n_on", 0, 300)

    mu_on  = None   # TODO: RooFormulaVar  s + b
    p_on   = None   # TODO: RooPoisson(n_on, mu_on)
    constr = None   # TODO: RooGaussian(b0, b, sb)       <- the constraint term
    pdf    = None   # TODO: RooProdPdf of p_on and constr

    data = ROOT.RooDataSet("d", "d", ROOT.RooArgSet(n_on))   # only n_on: b0 is NOT in the data
    n_on.setVal(non)
    data.add(ROOT.RooArgSet(n_on))
    nll = pdf.createNLL(data, ROOT.RooFit.GlobalObservables(ROOT.RooArgSet(b0)))
    return dict(locals())

## A2 · The significance
Implement `z0(m)`: free fit, fit with $s=0$, then $Z=\\sqrt{q_0}$.

In [ ]:
def minimise(nll):
    mn = ROOT.RooMinimizer(nll)
    mn.setPrintLevel(-1)
    mn.minimize("Minuit2", "Migrad")
    return nll.getVal()

def z0(m):
    """Significance of the background-only test:  Z = sqrt(q0),  q0 = 2 [ -lnL(s=0, b^^) + lnL(s^, b^) ]."""
    s, b, nll = m["s"], m["b"], m["nll"]
    # TODO 1: free fit (s and b floating) -> nll_free
    # TODO 2: fix s = 0 (s.setVal(0); s.setConstant(True)), re-fit b -> nll_null
    # TODO 3: release s (s.setConstant(False)) and return sqrt(max(2*(nll_null - nll_free), 0))
    return None

In [ ]:
non, noff, tau_val = 17, 40, 10.53
b0_val, sigma_b = noff / tau_val, np.sqrt(noff) / tau_val       # 3.80 +- 0.60

m_pois  = build_poisson_aux(non, noff, tau_val)
m_gauss = build_gauss(non, b0_val, sigma_b)

Zp, Zg = z0(m_pois), z0(m_gauss)
print("Z0 auxiliary Poisson : %.2f" % Zp)
print("Z0 Gaussian constraint: %.2f" % Zg)

In [ ]:
check("z0 implemented", Zp is not None and Zg is not None)
check("Poisson-aux significance about 4.6", abs(Zp - 4.57) < 0.15)
check("Gaussian constraint agrees when n_off is large", abs(Zg - Zp) < 0.15)

## A3 · A poor control region
Same $\\hat b=3.8$, but measured with only $n_{\\rm off}=4$ events ($\\tau=1.053$): the Gaussian constraint now has $\\sigma_b=1.9$.
Which model gives the larger significance? Which one is right?

In [ ]:
noff2, tau2 = 4, 1.053
m_pois2  = build_poisson_aux(non, noff2, tau2)
m_gauss2 = build_gauss(non, noff2 / tau2, np.sqrt(noff2) / tau2)
Zp2, Zg2 = z0(m_pois2), z0(m_gauss2)
print("poor control region:  Poisson aux Z = %.2f   Gaussian Z = %.2f" % (Zp2, Zg2))

In [ ]:
check("the Poisson auxiliary measurement gives about 3.1", abs(Zp2 - 3.06) < 0.2)
check("the Gaussian constraint is too optimistic here", Zg2 > Zp2 + 0.2)

## Questions (2–3 lines each)
1. Why do the two models agree for $n_{\\rm off}=40$ and not for $n_{\\rm off}=4$?
2. In the Gaussian model, what is $b_0$ and why is it **not** in the dataset? What would change in a toy study?
3. You read “background $0.5\\pm 0.4$” in a paper and have no control data. Which constraint would you use, and why not a Gaussian?